<a href="https://colab.research.google.com/github/ZiadMAlsawy/BigData/blob/main/mini%20Project%202/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark Case Study: Flight Delay & Cancellation Analysis (2019-2023)

**Dataset:** US Flight Delay & Cancellation Data (Kaggle - patrickzel/flight-delay-and-cancellation-dataset-2019-2023)

**Size:** 3,000,000 records x 32 columns

---

### Research Question
**What drives flight delay severity across US airlines and routes, and how do delay causes propagate over time?**

We investigate:
1. **Trend analysis** - Temporal delay patterns (moving averages, cumulative cancellations)
2. **Anomaly detection** - Routes/airlines with statistically abnormal delays
3. **Root-cause attribution** - Which delay category (carrier, weather, NAS, security, late aircraft) dominates per airline

### Why this dataset is suitable for distributed processing
- **3M rows** exceed single-node memory-efficient processing for complex window/join workloads
- **32 mixed-type columns** (dates, numerics, categoricals, nullable delay codes) exercise all Catalyst rules
- Natural support for **join optimization** via small lookup tables (airlines, airports)
- Rich **time dimension** enables window functions (moving averages, cumulative sums, ranking)

## 1. Environment Setup

In [1]:
import os, time, shutil
from pathlib import Path
import sys
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel
import shutil
from pathlib import Path
import kagglehub

In [2]:
print('PySpark version:', pyspark.__version__)
JAVA_OPTS = '-Djava.security.manager=allow'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f'--conf spark.driver.extraJavaOptions={JAVA_OPTS} '
    f'--conf spark.executor.extraJavaOptions={JAVA_OPTS} '
    'pyspark-shell'
)

PySpark version: 4.0.2


In [ ]:
# Build Spark session. Local cluster, 4 executors via [4], 4GB driver memory, 
# 200 shuffle partitions, adaptive query execution enabled, and broadcast join threshold set to 10MB.
spark = (SparkSession.builder
         .appName('FlightDelayCaseStudy')
         .master('local[4]')
         .config('spark.sql.shuffle.partitions', '200')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.autoBroadcastJoinThreshold', 10 * 1024 * 1024)
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel('WARN')
print('Spark UI:', sc.uiWebUrl)
print('Default parallelism:', sc.defaultParallelism)

Spark UI: http://d25a97fc8027:4040
Default parallelism: 4


## 2. Dataset Loading & Schema

We define the schema explicitly - this skips Spark's schema-inference scan and makes the benchmarks fair.

In [4]:
TARGET = Path.cwd() / 'flights_sample_3m.csv'

if TARGET.exists():
    print(f'{TARGET.name} already present - skipping download.')
    raise SystemExit(0)

cache_path = Path(kagglehub.dataset_download(
    'patrickzel/flight-delay-and-cancellation-dataset-2019-2023'))
src = cache_path / 'flights_sample_3m.csv'
if not src.exists():
    src = next(cache_path.rglob('flights_sample_3m.csv'))

shutil.copy(src, TARGET)
print(f'Copied dataset to {TARGET}')

100%|██████████| 140M/140M [00:07<00:00, 19.7MB/s]

Extracting files...


Copied dataset to /content/flights_sample_3m.csv


In [5]:
CSV_PATH = 'flights_sample_3m.csv'
PARQUET_PATH = 'flights.parquet'

schema = StructType([
    StructField('FL_DATE', DateType(), True),
    StructField('AIRLINE', StringType(), True),
    StructField('AIRLINE_DOT', StringType(), True),
    StructField('AIRLINE_CODE', StringType(), True),
    StructField('DOT_CODE', IntegerType(), True),
    StructField('FL_NUMBER', IntegerType(), True),
    StructField('ORIGIN', StringType(), True),
    StructField('ORIGIN_CITY', StringType(), True),
    StructField('DEST', StringType(), True),
    StructField('DEST_CITY', StringType(), True),
    StructField('CRS_DEP_TIME', IntegerType(), True),
    StructField('DEP_TIME', DoubleType(), True),
    StructField('DEP_DELAY', DoubleType(), True),
    StructField('TAXI_OUT', DoubleType(), True),
    StructField('WHEELS_OFF', DoubleType(), True),
    StructField('WHEELS_ON', DoubleType(), True),
    StructField('TAXI_IN', DoubleType(), True),
    StructField('CRS_ARR_TIME', IntegerType(), True),
    StructField('ARR_TIME', DoubleType(), True),
    StructField('ARR_DELAY', DoubleType(), True),
    StructField('CANCELLED', DoubleType(), True),
    StructField('CANCELLATION_CODE', StringType(), True),
    StructField('DIVERTED', DoubleType(), True),
    StructField('CRS_ELAPSED_TIME', DoubleType(), True),
    StructField('ELAPSED_TIME', DoubleType(), True),
    StructField('AIR_TIME', DoubleType(), True),
    StructField('DISTANCE', DoubleType(), True),
    StructField('DELAY_DUE_CARRIER', DoubleType(), True),
    StructField('DELAY_DUE_WEATHER', DoubleType(), True),
    StructField('DELAY_DUE_NAS', DoubleType(), True),
    StructField('DELAY_DUE_SECURITY', DoubleType(), True),
    StructField('DELAY_DUE_LATE_AIRCRAFT', DoubleType(), True),
])

flights = (spark.read
           .option('header', 'true')
           .schema(schema)
           .csv(CSV_PATH))

# Derived columns used across multiple queries
flights = (flights
           .withColumn('YEAR', F.year('FL_DATE'))
           .withColumn('MONTH', F.month('FL_DATE'))
           .withColumn('DAY_OF_WEEK', F.dayofweek('FL_DATE')))

flights.printSchema()
print('Total records:', flights.count())

root
 |-- FL_DATE: date (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- AIRLINE_DOT: string (nullable = true)
 |-- AIRLINE_CODE: string (nullable = true)
 |-- DOT_CODE: integer (nullable = true)
 |-- FL_NUMBER: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)

In [6]:
# Register as SQL temp view for the SQL API implementations
flights.createOrReplaceTempView('flights')
spark.sql('SELECT COUNT(*) AS n FROM flights').show()

+-------+
|      n|
+-------+
|3000000|
+-------+



### Build secondary lookup tables (for join demonstrations)

We derive two side tables from the main dataset:
- **airlines_dim** - small table (20 rows) - better for **broadcast join**
- **airport_stats** - medium table (360 rows) - can demonstrate **sort-merge** when broadcast is disabled

In [7]:
airlines_dim = (flights.select('AIRLINE_CODE', 'AIRLINE', 'DOT_CODE')
                .distinct())
airlines_dim.createOrReplaceTempView('airlines_dim')
print('airlines_dim rows:', airlines_dim.count())

airport_stats = (flights.groupBy('ORIGIN')
                 .agg(F.count('*').alias('ORIGIN_FLIGHT_COUNT'),
                      F.first('ORIGIN_CITY').alias('ORIGIN_CITY')))
airport_stats.createOrReplaceTempView('airport_stats')
print('airport_stats rows:', airport_stats.count())

airlines_dim rows: 18
airport_stats rows: 380


### Benchmark helper

A single utility records execution time and triggers an action so lazy transformations actually execute.

In [8]:
PERF = []
def time_it(label, api, fn):
    t0 = time.perf_counter()
    result = fn()
    if hasattr(result, 'count'):
        n = result.count()
    else:
        n = int(result)
    elapsed = time.perf_counter() - t0
    PERF.append({'query': label, 'api': api, 'seconds': round(elapsed, 3), 'rows': n})
    print(f'[{label:25s}][{api:10s}] {elapsed:.3f}s  rows={n}')
    return result

---

## 3. Queries (12) - implemented in RDD, DataFrame, and Spark SQL

For each query we:
1. Describe the analytical intent.
2. Implement it **three ways**.
3. Benchmark execution time.
4. Print `.explain(True)` for the DataFrame version - includes **parsed, analyzed, optimized logical, and physical** plans.

In [9]:
# Shared RDD used by every RDD-based query.
flights_rdd = flights.rdd
flights_rdd.cache()

MapPartitionsRDD[39] at javaToPython at NativeMethodAccessorImpl.java:0

### Q1. Filtering with complex conditions

> Find flights in **winter months (Dec/Jan/Feb)** that were **delayed >= 60 minutes at arrival**, **not cancelled**, and had **distance > 500 miles**.

In [ ]:
# RDD API
def q1_rdd():
    return flights_rdd.filter(lambda r: r.MONTH in (12, 1, 2)
                                     and r.ARR_DELAY is not None and r.ARR_DELAY >= 60
                                     and r.CANCELLED == 0.0
                                     and r.DISTANCE is not None and r.DISTANCE > 500)
time_it('Q1_filter', 'RDD', lambda: q1_rdd().count())

# DataFrame API
def q1_df():
    return (flights
            .filter(F.col('MONTH').isin(12, 1, 2))
            .filter(F.col('ARR_DELAY') >= 60)
            .filter(F.col('CANCELLED') == 0.0)
            .filter(F.col('DISTANCE') > 500))
q1_df_res = time_it('Q1_filter', 'DataFrame', q1_df)

# Spark SQL
def q1_sql():
    return spark.sql("""
        SELECT * FROM flights
        WHERE MONTH IN (12,1,2)
          AND ARR_DELAY >= 60
          AND CANCELLED = 0.0
          AND DISTANCE > 500
    """)
time_it('Q1_filter', 'SQL', q1_sql)

print('\n=== Q1 DataFrame .explain(True) ===')
q1_df_res.explain(True)

[Q1_filter                ][RDD       ] 36.832s  rows=28379
[Q1_filter                ][DataFrame ] 10.848s  rows=28379
[Q1_filter                ][SQL       ] 11.997s  rows=28379

=== Q1 DataFrame .explain(True) ===
== Parsed Logical Plan ==
'Filter '`>`('DISTANCE, 500)
+- Filter (CANCELLED#20 = 0.0)
   +- Filter (ARR_DELAY#19 >= cast(60 as double))
      +- Filter MONTH#34 IN (12,1,2)
         +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION_CODE#21, DIVERTED#22, CRS_ELAPSED_TIME#23, ELAPSED_TIME#24, ... 10 more fields]
            +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#

### Q2. Aggregations (SUM, AVG, COUNT, MAX, MIN)

> Aggregate delay statistics per airline: total flights, cancellation count, avg / min / max arrival delay.

In [ ]:
# RDD API
def q2_rdd():
    kv = flights_rdd.map(lambda r: (r.AIRLINE_CODE, (
        1,
        float(r.CANCELLED or 0),
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else 0.0,
        1 if r.ARR_DELAY is not None else 0,
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else float('-inf'),
        float(r.ARR_DELAY) if r.ARR_DELAY is not None else float('inf'))))
    reduced = kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2], a[3]+b[3], max(a[4], b[4]), min(a[5], b[5])))
    return reduced.map(lambda x: (x[0], x[1][0], x[1][1],
                                  x[1][2]/x[1][3] if x[1][3] else None, x[1][4], x[1][5]))
time_it('Q2_aggregate', 'RDD', lambda: q2_rdd().count())

# DataFrame API
def q2_df():
    return (flights.groupBy('AIRLINE_CODE')
            .agg(F.count('*').alias('total_flights'),
                 F.sum('CANCELLED').alias('cancellations'),
                 F.avg('ARR_DELAY').alias('avg_arr_delay'),
                 F.max('ARR_DELAY').alias('max_arr_delay'),
                 F.min('ARR_DELAY').alias('min_arr_delay')))
q2_df_res = time_it('Q2_aggregate', 'DataFrame', q2_df)

# Spark SQL
def q2_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE,
               COUNT(*)          AS total_flights,
               SUM(CANCELLED)    AS cancellations,
               AVG(ARR_DELAY)    AS avg_arr_delay,
               MAX(ARR_DELAY)    AS max_arr_delay,
               MIN(ARR_DELAY)    AS min_arr_delay
        FROM flights
        GROUP BY AIRLINE_CODE
    """)
time_it('Q2_aggregate', 'SQL', q2_sql)

q2_df_res.orderBy(F.desc('total_flights')).show(truncate=False)
print('\n=== Q2 DataFrame .explain(True) ===')
q2_df_res.explain(True)

[Q2_aggregate             ][RDD       ] 65.500s  rows=18
[Q2_aggregate             ][DataFrame ] 9.774s  rows=18
[Q2_aggregate             ][SQL       ] 7.547s  rows=18
+------------+-------------+-------------+-------------------+-------------+-------------+
|AIRLINE_CODE|total_flights|cancellations|avg_arr_delay      |max_arr_delay|min_arr_delay|
+------------+-------------+-------------+-------------------+-------------+-------------+
|WN          |576470       |19465.0      |3.2697955813330117 |697.0        |-82.0        |
|DL          |395239       |5982.0       |1.0850788339017954 |1241.0       |-86.0        |
|AA          |383106       |10907.0      |6.661228711969786  |2934.0       |-96.0        |
|OO          |343737       |7745.0       |4.023311421969874  |2308.0       |-96.0        |
|UA          |254504       |5536.0       |5.035985016312885  |1458.0       |-81.0        |
|YX          |143107       |4646.0       |0.5901756824252427 |1263.0       |-88.0        |
|MQ         

### Q3. Grouping by multiple attributes

> Average arrival delay per **(airline, origin airport, month)** - useful for hotspot detection.

In [ ]:
# RDD API
def q3_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.AIRLINE_CODE, r.ORIGIN, r.MONTH), (float(r.ARR_DELAY), 1))))
    return (kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
              .mapValues(lambda v: v[0]/v[1]))
time_it('Q3_multi_group', 'RDD', lambda: q3_rdd().count())

# DataFrame API
def q3_df():
    return (flights.groupBy('AIRLINE_CODE', 'ORIGIN', 'MONTH')
            .agg(F.avg('ARR_DELAY').alias('avg_arr_delay'),
                 F.count('*').alias('n')))
q3_df_res = time_it('Q3_multi_group', 'DataFrame', q3_df)

# Spark SQL
def q3_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE, ORIGIN, MONTH,
               AVG(ARR_DELAY) AS avg_arr_delay,
               COUNT(*)       AS n
        FROM flights
        WHERE ARR_DELAY IS NOT NULL
        GROUP BY AIRLINE_CODE, ORIGIN, MONTH
    """)
time_it('Q3_multi_group', 'SQL', q3_sql)

print('\n=== Q3 DataFrame .explain(True) ===')
q3_df_res.explain(True)

[Q3_multi_group           ][RDD       ] 54.288s  rows=24911
[Q3_multi_group           ][DataFrame ] 13.896s  rows=24939
[Q3_multi_group           ][SQL       ] 14.507s  rows=24911

=== Q3 DataFrame .explain(True) ===
== Parsed Logical Plan ==
'Aggregate ['AIRLINE_CODE, 'ORIGIN, 'MONTH], ['AIRLINE_CODE, 'ORIGIN, 'MONTH, 'avg('ARR_DELAY) AS avg_arr_delay#465, 'count(*) AS n#466]
+- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION_CODE#21, DIVERTED#22, CRS_ELAPSED_TIME#23, ELAPSED_TIME#24, ... 10 more fields]
   +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16

### Q4. Sorting and ranking - Top 20 most-delayed routes

In [ ]:
# RDD API
def q4_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.ORIGIN, r.DEST), (float(r.ARR_DELAY), 1))))
    agg = kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda v: v[0]/v[1])
    return agg.takeOrdered(20, key=lambda x: -x[1])
time_it('Q4_top20_routes', 'RDD', lambda: len(q4_rdd()))

# DataFrame API
def q4_df():
    return (flights.groupBy('ORIGIN', 'DEST')
            .agg(F.avg('ARR_DELAY').alias('avg_delay'),
                 F.count('*').alias('flights'))
            .filter('flights > 100')
            .orderBy(F.desc('avg_delay'))
            .limit(20))
q4_df_res = time_it('Q4_top20_routes', 'DataFrame', q4_df)

# Spark SQL
def q4_sql():
    return spark.sql("""
        SELECT ORIGIN, DEST,
               AVG(ARR_DELAY) AS avg_delay,
               COUNT(*)       AS flights
        FROM flights
        GROUP BY ORIGIN, DEST
        HAVING COUNT(*) > 100
        ORDER BY avg_delay DESC
        LIMIT 20
    """)
time_it('Q4_top20_routes', 'SQL', q4_sql)

q4_df_res.show(20, truncate=False)
print('\n=== Q4 DataFrame .explain(True) ===')
q4_df_res.explain(True)

[Q4_top20_routes          ][RDD       ] 49.753s  rows=20
[Q4_top20_routes          ][DataFrame ] 12.398s  rows=20
[Q4_top20_routes          ][SQL       ] 11.207s  rows=20
+------+----+------------------+-------+
|ORIGIN|DEST|avg_delay         |flights|
+------+----+------------------+-------+
|RNO   |JFK |47.41237113402062 |101    |
|ABQ   |JFK |40.10091743119266 |116    |
|ACK   |LGA |38.88059701492537 |145    |
|ASE   |ORD |38.53611111111111 |407    |
|BDL   |SJU |37.03666666666667 |307    |
|ONT   |JFK |36.376068376068375|124    |
|IAD   |SHD |34.970588235294116|140    |
|SJU   |DCA |33.23841059602649 |153    |
|ASE   |DFW |33.1972972972973  |411    |
|CHO   |ORD |33.0              |142    |
|ASE   |SFO |31.689393939393938|144    |
|SAN   |FLL |31.05982905982906 |123    |
|SRQ   |CVG |30.75             |112    |
|ASE   |IAH |30.53170731707317 |230    |
|EYW   |LGA |30.03669724770642 |112    |
|PGD   |AVL |29.934579439252335|109    |
|SFB   |FNT |29.698924731182796|101    |
|PGD   |F

### Q5. Window function - 7-day moving average of arrival delay per airline

Demonstrates `ROWS BETWEEN N PRECEDING AND CURRENT ROW` - a key trend-analysis tool.

In [ ]:
# RDD API (manual sliding window after groupByKey)
def q5_rdd():
    daily = (flights_rdd
             .filter(lambda r: r.ARR_DELAY is not None)
             .map(lambda r: ((r.AIRLINE_CODE, r.FL_DATE), (float(r.ARR_DELAY), 1)))
             .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
             .map(lambda x: (x[0][0], (x[0][1], x[1][0]/x[1][1]))))
    grouped = daily.groupByKey().mapValues(lambda it: sorted(it))
    def rolling(pairs):
        out = []
        for i in range(len(pairs)):
            window = pairs[max(0, i-6):i+1]
            out.append((pairs[i][0], sum(p[1] for p in window)/len(window)))
        return out
    return grouped.flatMapValues(rolling)
time_it('Q5_moving_avg', 'RDD', lambda: q5_rdd().count())

# DataFrame API
def q5_df():
    daily = (flights.filter(F.col('ARR_DELAY').isNotNull())
             .groupBy('AIRLINE_CODE', 'FL_DATE')
             .agg(F.avg('ARR_DELAY').alias('daily_avg')))
    w = Window.partitionBy('AIRLINE_CODE').orderBy('FL_DATE').rowsBetween(-6, 0)
    return daily.withColumn('ma7', F.avg('daily_avg').over(w))
q5_df_res = time_it('Q5_moving_avg', 'DataFrame', q5_df)

# Spark SQL
def q5_sql():
    return spark.sql("""
        WITH daily AS (
          SELECT AIRLINE_CODE, FL_DATE, AVG(ARR_DELAY) AS daily_avg
          FROM flights WHERE ARR_DELAY IS NOT NULL
          GROUP BY AIRLINE_CODE, FL_DATE
        )
        SELECT AIRLINE_CODE, FL_DATE, daily_avg,
               AVG(daily_avg) OVER (PARTITION BY AIRLINE_CODE
                                    ORDER BY FL_DATE
                                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
        FROM daily
    """)
time_it('Q5_moving_avg', 'SQL', q5_sql)

q5_df_res.show(10)
print('\n=== Q5 DataFrame .explain(True) ===')
q5_df_res.explain(True)

[Q5_moving_avg            ][RDD       ] 54.472s  rows=28336
[Q5_moving_avg            ][DataFrame ] 13.827s  rows=28336
[Q5_moving_avg            ][SQL       ] 13.094s  rows=28336
+------------+----------+-------------------+-------------------+
|AIRLINE_CODE|   FL_DATE|          daily_avg|                ma7|
+------------+----------+-------------------+-------------------+
|          9E|2019-01-01| -7.859649122807017| -7.859649122807017|
|          9E|2019-01-02|  -6.47457627118644| -7.167112696996728|
|          9E|2019-01-03| -12.56338028169014|   -8.9658685585612|
|          9E|2019-01-04| -8.033333333333333| -8.732734752254233|
|          9E|2019-01-05|             -14.16| -9.818187801803386|
|          9E|2019-01-06|-11.318181818181818|-10.068186804533125|
|          9E|2019-01-07| -9.405797101449275| -9.973559704092576|
|          9E|2019-01-08| -9.225352112676056|-10.168660131216724|
|          9E|2019-01-09| -2.486111111111111| -9.598879394063106|
|          9E|2019-01-10| 14

### Q6. Window function - Cumulative cancellations per airline over time

In [ ]:
# RDD API
def q6_rdd():
    daily = (flights_rdd
             .map(lambda r: ((r.AIRLINE_CODE, r.FL_DATE), float(r.CANCELLED or 0)))
             .reduceByKey(lambda a, b: a + b)
             .map(lambda x: (x[0][0], (x[0][1], x[1]))))
    grouped = daily.groupByKey().mapValues(lambda it: sorted(it))
    def cumul(pairs):
        total = 0.0; out = []
        for d, c in pairs:
            total += c; out.append((d, total))
        return out
    return grouped.flatMapValues(cumul)
time_it('Q6_cumulative', 'RDD', lambda: q6_rdd().count())

# DataFrame API
def q6_df():
    daily = (flights.groupBy('AIRLINE_CODE', 'FL_DATE')
             .agg(F.sum('CANCELLED').alias('daily_cancel')))
    w = (Window.partitionBy('AIRLINE_CODE').orderBy('FL_DATE')
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    return daily.withColumn('cum_cancel', F.sum('daily_cancel').over(w))
q6_df_res = time_it('Q6_cumulative', 'DataFrame', q6_df)

# Spark SQL
def q6_sql():
    return spark.sql("""
        WITH daily AS (
          SELECT AIRLINE_CODE, FL_DATE, SUM(CANCELLED) AS daily_cancel
          FROM flights GROUP BY AIRLINE_CODE, FL_DATE
        )
        SELECT AIRLINE_CODE, FL_DATE, daily_cancel,
               SUM(daily_cancel) OVER (PARTITION BY AIRLINE_CODE
                                       ORDER BY FL_DATE
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cancel
        FROM daily
    """)
time_it('Q6_cumulative', 'SQL', q6_sql)

print('\n=== Q6 DataFrame .explain(True) ===')
q6_df_res.explain(True)

[Q6_cumulative            ][RDD       ] 49.895s  rows=28355
[Q6_cumulative            ][DataFrame ] 12.091s  rows=28355
[Q6_cumulative            ][SQL       ] 11.837s  rows=28355

=== Q6 DataFrame .explain(True) ===
== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(cum_cancel, 'sum('daily_cancel) windowspecdefinition('AIRLINE_CODE, 'FL_DATE ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())), None)]
+- Aggregate [AIRLINE_CODE#3, FL_DATE#0], [AIRLINE_CODE#3, FL_DATE#0, sum(CANCELLED#20) AS daily_cancel#740]
   +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION_CODE#21, DIVERTED#22, CRS_ELAPSED_TIME#23, ELAPSED_TIME#24, ... 10 more fields]
      +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#

### Q7. Window function - Rank airlines by on-time rate per month

In [ ]:
# RDD API
def q7_rdd():
    kv = (flights_rdd
          .filter(lambda r: r.ARR_DELAY is not None)
          .map(lambda r: ((r.YEAR, r.MONTH, r.AIRLINE_CODE),
                          (1, 1 if r.ARR_DELAY <= 15 else 0))))
    agg = (kv.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
              .map(lambda x: (x[0][0], x[0][1], x[0][2], x[1][1]/x[1][0])))
    grouped = agg.map(lambda x: ((x[0], x[1]), (x[2], x[3]))).groupByKey()
    def rank(rows):
        s = sorted(rows, key=lambda r: -r[1])
        return [(airline, rate, i+1) for i, (airline, rate) in enumerate(s)]
    return grouped.flatMapValues(rank)
time_it('Q7_rank', 'RDD', lambda: q7_rdd().count())

# DataFrame API
def q7_df():
    monthly = (flights.filter(F.col('ARR_DELAY').isNotNull())
               .groupBy('YEAR', 'MONTH', 'AIRLINE_CODE')
               .agg((F.sum((F.col('ARR_DELAY') <= 15).cast('int')) / F.count('*')).alias('on_time_rate')))
    w = Window.partitionBy('YEAR', 'MONTH').orderBy(F.desc('on_time_rate'))
    return monthly.withColumn('rnk', F.rank().over(w))
q7_df_res = time_it('Q7_rank', 'DataFrame', q7_df)

# Spark SQL
def q7_sql():
    return spark.sql("""
        WITH monthly AS (
          SELECT YEAR, MONTH, AIRLINE_CODE,
                 SUM(CASE WHEN ARR_DELAY <= 15 THEN 1 ELSE 0 END) / COUNT(*) AS on_time_rate
          FROM flights WHERE ARR_DELAY IS NOT NULL
          GROUP BY YEAR, MONTH, AIRLINE_CODE
        )
        SELECT YEAR, MONTH, AIRLINE_CODE, on_time_rate,
               RANK() OVER (PARTITION BY YEAR, MONTH ORDER BY on_time_rate DESC) AS rnk
        FROM monthly
    """)
time_it('Q7_rank', 'SQL', q7_sql)

q7_df_res.orderBy('YEAR', 'MONTH', 'rnk').show(15)
print('\n=== Q7 DataFrame .explain(True) ===')
q7_df_res.explain(True)

[Q7_rank                  ][RDD       ] 53.337s  rows=933
[Q7_rank                  ][DataFrame ] 11.854s  rows=933
[Q7_rank                  ][SQL       ] 10.806s  rows=933
+----+-----+------------+------------------+---+
|YEAR|MONTH|AIRLINE_CODE|      on_time_rate|rnk|
+----+-----+------------+------------------+---+
|2019|    1|          DL| 0.880646443237515|  1|
|2019|    1|          HA|0.8712574850299402|  2|
|2019|    1|          WN|0.8538291605301914|  3|
|2019|    1|          OH|0.8490981082270127|  4|
|2019|    1|          AS|0.8368550368550368|  5|
|2019|    1|          AA|0.8293284842187297|  6|
|2019|    1|          NK| 0.823841059602649|  7|
|2019|    1|          YV|0.8166109253065775|  8|
|2019|    1|          9E|0.8124690747154873|  9|
|2019|    1|          UA| 0.805152630439489| 10|
|2019|    1|          OO|0.7828916045072211| 11|
|2019|    1|          F9|0.7765306122448979| 12|
|2019|    1|          MQ|0.7746536267318663| 13|
|2019|    1|          YX| 0.77337110481586

### Q8. Nested / subquery - airlines with above-overall-average arrival delay

In [ ]:
# RDD API
def q8_rdd():
    non_null = (flights_rdd
                .filter(lambda r: r.ARR_DELAY is not None)
                .map(lambda r: (r.AIRLINE_CODE, float(r.ARR_DELAY))))
    total = non_null.map(lambda x: (x[1], 1)).reduce(lambda a, b: (a[0]+b[0], a[1]+b[1]))
    overall = total[0] / total[1]
    per_airline = (non_null.map(lambda x: (x[0], (x[1], 1)))
                            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
                            .mapValues(lambda v: v[0]/v[1]))
    return per_airline.filter(lambda x: x[1] > overall)
time_it('Q8_subquery', 'RDD', lambda: q8_rdd().count())

# DataFrame API
def q8_df():
    overall = flights.agg(F.avg('ARR_DELAY').alias('o')).first()['o']
    return (flights.groupBy('AIRLINE_CODE')
                   .agg(F.avg('ARR_DELAY').alias('avg_delay'))
                   .filter(F.col('avg_delay') > F.lit(overall)))
q8_df_res = time_it('Q8_subquery', 'DataFrame', q8_df)

# Spark SQL
def q8_sql():
    return spark.sql("""
        SELECT AIRLINE_CODE, AVG(ARR_DELAY) AS avg_delay
        FROM flights
        GROUP BY AIRLINE_CODE
        HAVING AVG(ARR_DELAY) > (SELECT AVG(ARR_DELAY) FROM flights)
    """)
q8_sql_res = time_it('Q8_subquery', 'SQL', q8_sql)

q8_sql_res.orderBy(F.desc('avg_delay')).show()
print('\n=== Q8 SQL .explain(True) ===')
q8_sql_res.explain(True)

[Q8_subquery              ][RDD       ] 83.325s  rows=8
[Q8_subquery              ][DataFrame ] 16.736s  rows=8
[Q8_subquery              ][SQL       ] 16.295s  rows=8
+------------+------------------+
|AIRLINE_CODE|         avg_delay|
+------------+------------------+
|          G4|13.284601127961896|
|          B6|12.276124516889453|
|          F9|11.100428951858525|
|          EV| 10.03197593448833|
|          NK| 8.029484978540772|
|          YV|7.3098100100837105|
|          AA| 6.661228711969786|
|          UA| 5.035985016312885|
+------------+------------------+


=== Q8 SQL .explain(True) ===
== Parsed Logical Plan ==
'UnresolvedHaving ('AVG('ARR_DELAY) > scalar-subquery#1021 [])
:  +- 'Project [unresolvedalias('AVG('ARR_DELAY))]
:     +- 'UnresolvedRelation [flights], [], false
+- 'Aggregate ['AIRLINE_CODE], ['AIRLINE_CODE, 'AVG('ARR_DELAY) AS avg_delay#1020]
   +- 'UnresolvedRelation [flights], [], false

== Analyzed Logical Plan ==
AIRLINE_CODE: string, avg_delay: double
Fil

### Q9. JOIN - **Broadcast join** with the small `airlines_dim` table

Because `airlines_dim` is well under the 10 MB broadcast threshold, Catalyst picks a **BroadcastHashJoin**. We also hint it explicitly.

In [ ]:
# RDD API (manual broadcast)
def q9_rdd():
    airlines_map = dict((r['AIRLINE_CODE'], r['AIRLINE']) for r in airlines_dim.collect())
    bcast = sc.broadcast(airlines_map)
    return (flights_rdd
            .filter(lambda r: r.ARR_DELAY is not None)
            .map(lambda r: (bcast.value.get(r.AIRLINE_CODE, 'UNKNOWN'), (float(r.ARR_DELAY), 1)))
            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
            .mapValues(lambda v: v[0]/v[1]))
time_it('Q9_broadcast_join', 'RDD', lambda: q9_rdd().count())

# DataFrame API (hint broadcast)}
def q9_df():
    f = flights.alias("f")
    a = airlines_dim.alias("a")
    return (
        f.join(F.broadcast(a), 'AIRLINE_CODE')
         .groupBy(F.col('a.AIRLINE'))
         .agg(F.avg('f.ARR_DELAY').alias('avg_delay'))
    )
q9_df_res = time_it('Q9_broadcast_join', 'DataFrame', q9_df)

# Spark SQL (hint broadcast)
def q9_sql():
    return spark.sql("""
        SELECT /*+ BROADCAST(a) */ a.AIRLINE, AVG(f.ARR_DELAY) AS avg_delay
        FROM flights f JOIN airlines_dim a ON f.AIRLINE_CODE = a.AIRLINE_CODE
        GROUP BY a.AIRLINE
    """)
time_it('Q9_broadcast_join', 'SQL', q9_sql)

print('\n=== Q9 DataFrame .explain(True) - expect BroadcastHashJoin ===')
q9_df_res.explain(True)

[Q9_broadcast_join        ][RDD       ] 43.106s  rows=18
[Q9_broadcast_join        ][DataFrame ] 19.617s  rows=18
[Q9_broadcast_join        ][SQL       ] 21.088s  rows=18

=== Q9 DataFrame .explain(True) - expect BroadcastHashJoin ===
== Parsed Logical Plan ==
'Aggregate ['a.AIRLINE], ['a.AIRLINE, 'avg('f.ARR_DELAY) AS avg_delay#445]
+- Project [AIRLINE_CODE#3, FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION_CODE#21, DIVERTED#22, CRS_ELAPSED_TIME#23, ELAPSED_TIME#24, ... 12 more fields]
   +- Join Inner, (AIRLINE_CODE#3 = AIRLINE_CODE#412)
      :- SubqueryAlias f
      :  +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13

### Q10. JOIN - **Sort-Merge join** by disabling broadcast

Disabling auto-broadcast forces SMJ. Same logical join, very different physical plan - **SortMergeJoin** with two full shuffles.

In [18]:
# Capture the SMJ plan once

def bench_q10_df():
    f = flights.alias("f")
    a = airport_stats.alias("a")
    prev = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
    try:
        return (f.join(a,F.col("f.ORIGIN") == F.col('a.ORIGIN'))
                       .groupBy(F.col('a.ORIGIN_CITY'))
                       .agg(F.avg(F.col('f.ARR_DELAY')).alias('avg_delay')))
    finally:
        spark.conf.set('spark.sql.autoBroadcastJoinThreshold', prev)
time_it('Q10_sortmerge_join', 'DataFrame', bench_q10_df)

def bench_q10_sql():
    prev = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
    try:
        return spark.sql("""
            SELECT /*+ MERGE(f, s) */ s.ORIGIN_CITY, AVG(f.ARR_DELAY) AS avg_delay
            FROM flights f JOIN airport_stats s ON f.ORIGIN = s.ORIGIN
            GROUP BY s.ORIGIN_CITY
        """)
    finally:
        spark.conf.set('spark.sql.autoBroadcastJoinThreshold', prev)
time_it('Q10_sortmerge_join', 'SQL', bench_q10_sql)

def bench_q10_rdd():
    left = (flights_rdd.filter(lambda r: r.ARR_DELAY is not None)
                       .map(lambda r: (r.ORIGIN, float(r.ARR_DELAY))))
    right = airport_stats.rdd.map(lambda r: (r['ORIGIN'], r['ORIGIN_CITY']))
    joined = left.join(right)
    return (joined.map(lambda x: (x[1][1], (x[1][0], 1)))
                  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
                  .mapValues(lambda v: v[0]/v[1]))
time_it('Q10_sortmerge_join', 'RDD', lambda: bench_q10_rdd().count())

[Q10_sortmerge_join       ][DataFrame ] 32.963s  rows=372
[Q10_sortmerge_join       ][SQL       ] 27.132s  rows=372
[Q10_sortmerge_join       ][RDD       ] 72.439s  rows=372


372

### Q11. Delay root-cause attribution (complex aggregation)

> For each airline, which delay cause contributes the most minutes? (carrier / weather / NAS / security / late-aircraft)

In [ ]:
CAUSES = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
         'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

# RDD API
def q11_rdd():
    def row_to_kv(r):
        vals = tuple(float(getattr(r, c) or 0.0) for c in CAUSES)
        return (r.AIRLINE_CODE, vals)
    agg = (flights_rdd.map(row_to_kv)
                      .reduceByKey(lambda a, b: tuple(x+y for x, y in zip(a, b))))
    return agg.mapValues(lambda v: CAUSES[v.index(max(v))])
time_it('Q11_root_cause', 'RDD', lambda: q11_rdd().count())

# DataFrame API
def q11_df():
    sums = flights.groupBy('AIRLINE_CODE').agg(*[F.sum(c).alias(c) for c in CAUSES])
    expr = F.array(*[F.struct(F.col(c).alias('v'), F.lit(c).alias('k')) for c in CAUSES])
    return sums.withColumn('top_cause', F.array_max(expr).getField('k'))
q11_df_res = time_it('Q11_root_cause', 'DataFrame', q11_df)

# Spark SQL
def q11_sql():
    return spark.sql("""
        WITH sums AS (
          SELECT AIRLINE_CODE,
                 SUM(DELAY_DUE_CARRIER)       AS c_carrier,
                 SUM(DELAY_DUE_WEATHER)       AS c_weather,
                 SUM(DELAY_DUE_NAS)           AS c_nas,
                 SUM(DELAY_DUE_SECURITY)      AS c_security,
                 SUM(DELAY_DUE_LATE_AIRCRAFT) AS c_late
          FROM flights GROUP BY AIRLINE_CODE
        )
        SELECT AIRLINE_CODE,
               CASE greatest(c_carrier, c_weather, c_nas, c_security, c_late)
                 WHEN c_carrier  THEN 'CARRIER'
                 WHEN c_weather  THEN 'WEATHER'
                 WHEN c_nas      THEN 'NAS'
                 WHEN c_security THEN 'SECURITY'
                 ELSE 'LATE_AIRCRAFT'
               END AS top_cause
        FROM sums
    """)
time_it('Q11_root_cause', 'SQL', q11_sql)

q11_df_res.show(truncate=False)
print('\n=== Q11 DataFrame .explain(True) ===')
q11_df_res.explain(True)

[Q11_root_cause           ][RDD       ] 58.999s  rows=18
[Q11_root_cause           ][DataFrame ] 9.377s  rows=18
[Q11_root_cause           ][SQL       ] 8.698s  rows=18
+------------+-----------------+-----------------+-------------+------------------+-----------------------+-----------------------+
|AIRLINE_CODE|DELAY_DUE_CARRIER|DELAY_DUE_WEATHER|DELAY_DUE_NAS|DELAY_DUE_SECURITY|DELAY_DUE_LATE_AIRCRAFT|top_cause              |
+------------+-----------------+-----------------+-------------+------------------+-----------------------+-----------------------+
|UA          |1007314.0        |176197.0         |812830.0     |318.0             |1366533.0              |DELAY_DUE_LATE_AIRCRAFT|
|NK          |396139.0         |69691.0          |542057.0     |10310.0           |404300.0               |DELAY_DUE_NAS          |
|AA          |2004811.0        |287614.0         |926715.0     |12659.0           |2290610.0              |DELAY_DUE_LATE_AIRCRAFT|
|EV          |99441.0          |15776.0

### Q12. Anomaly detection - flights delayed > 3 sigma above their route's mean

In [ ]:
# DataFrame API
def q12_df():
    w = Window.partitionBy('ORIGIN', 'DEST')
    return (flights.filter(F.col('ARR_DELAY').isNotNull())
            .withColumn('route_mean', F.avg('ARR_DELAY').over(w))
            .withColumn('route_sd', F.stddev('ARR_DELAY').over(w))
            .filter(F.col('ARR_DELAY') > F.col('route_mean') + 3 * F.col('route_sd')))
q12_df_res = time_it('Q12_anomaly', 'DataFrame', q12_df)

# Spark SQL
def q12_sql():
    return spark.sql("""
        SELECT * FROM (
            SELECT FL_DATE, AIRLINE_CODE, ORIGIN, DEST, ARR_DELAY,
                   AVG(ARR_DELAY)    OVER (PARTITION BY ORIGIN, DEST) AS route_mean,
                   STDDEV(ARR_DELAY) OVER (PARTITION BY ORIGIN, DEST) AS route_sd
            FROM flights WHERE ARR_DELAY IS NOT NULL
        ) t WHERE ARR_DELAY > route_mean + 3 * route_sd
    """)
time_it('Q12_anomaly', 'SQL', q12_sql)

# RDD (2-pass)
def q12_rdd():
    pairs = (flights_rdd.filter(lambda r: r.ARR_DELAY is not None)
                        .map(lambda r: ((r.ORIGIN, r.DEST), float(r.ARR_DELAY))))
    stats = (pairs.map(lambda x: (x[0], (x[1], x[1]*x[1], 1)))
                  .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
                  .mapValues(lambda v: (v[0]/v[2], max(0, v[1]/v[2] - (v[0]/v[2])**2) ** 0.5)))
    stats_map = dict(stats.collect())
    bcast = sc.broadcast(stats_map)
    return pairs.filter(lambda x: (x[1] - bcast.value[x[0]][0]) > 3 * bcast.value[x[0]][1])
time_it('Q12_anomaly', 'RDD', lambda: q12_rdd().count())

print('\n=== Q12 DataFrame .explain(True) ===')
q12_df_res.explain(True)

[Q12_anomaly              ][DataFrame ] 20.861s  rows=52660
[Q12_anomaly              ][SQL       ] 17.733s  rows=52660
[Q12_anomaly              ][RDD       ] 95.801s  rows=52797

=== Q12 DataFrame .explain(True) ===
== Parsed Logical Plan ==
'Filter '`>`('ARR_DELAY, '`+`('route_mean, '`*`('route_sd, 3)))
+- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION_CODE#21, DIVERTED#22, CRS_ELAPSED_TIME#23, ELAPSED_TIME#24, ... 12 more fields]
   +- Project [FL_DATE#0, AIRLINE#1, AIRLINE_DOT#2, AIRLINE_CODE#3, DOT_CODE#4, FL_NUMBER#5, ORIGIN#6, ORIGIN_CITY#7, DEST#8, DEST_CITY#9, CRS_DEP_TIME#10, DEP_TIME#11, DEP_DELAY#12, TAXI_OUT#13, WHEELS_OFF#14, WHEELS_ON#15, TAXI_IN#16, CRS_ARR_TIME#17, ARR_TIME#18, ARR_DELAY#19, CANCELLED#20, CANCELLATION

---

## 4. Optimization & Analysis

### 4.1 Caching impact - run Q2 twice: cold vs. cached

In [24]:
# Cold run (uncached)
flights.unpersist()
t0 = time.perf_counter()
q2_df().count()
cold = time.perf_counter() - t0

# Cache then run
flights.cache()
flights.count()  # materialize cache

t0 = time.perf_counter()
q2_df().count()
warm = time.perf_counter() - t0

print(f'Cold: {cold:.3f}s,  Cached: {warm:.3f}s,  Speedup: {cold/warm:.2f}x')
PERF.append({'query': 'Q2_cache_cold', 'api': 'DataFrame', 'seconds': round(cold, 3), 'rows': None})
PERF.append({'query': 'Q2_cache_warm', 'api': 'DataFrame', 'seconds': round(warm, 3), 'rows': None})

Cold: 8.590s,  Cached: 1.500s,  Speedup: 5.73x


### 4.2 File format comparison - CSV vs. Parquet

We write the dataset as partitioned Parquet (partitioned by `YEAR`) and re-run Q3.

In [25]:
if not Path(PARQUET_PATH).exists():
    (flights.write
       .mode('overwrite')
       .partitionBy('YEAR')
       .parquet(PARQUET_PATH))
    print('Parquet written.')

flights_parq = spark.read.parquet(PARQUET_PATH)
flights_parq.createOrReplaceTempView('flights_parq')

# CSV-based Q3
t0 = time.perf_counter(); q3_df().count(); csv_t = time.perf_counter() - t0

# Parquet-based Q3
def q3_parq():
    return (flights_parq.groupBy('AIRLINE_CODE', 'ORIGIN', 'MONTH')
            .agg(F.avg('ARR_DELAY').alias('avg_arr_delay')))
t0 = time.perf_counter(); q3_parq().count(); parq_t = time.perf_counter() - t0

print(f'CSV Q3:     {csv_t:.3f}s')
print(f'Parquet Q3: {parq_t:.3f}s   (speedup {csv_t/parq_t:.2f}x)')
PERF.append({'query': 'Q3_format_csv', 'api': 'DataFrame', 'seconds': round(csv_t, 3), 'rows': None})
PERF.append({'query': 'Q3_format_parquet', 'api': 'DataFrame', 'seconds': round(parq_t, 3), 'rows': None})

Parquet written.
CSV Q3:     2.839s
Parquet Q3: 3.137s   (speedup 0.91x)


### 4.3 Partition pruning on the partitioned Parquet

When we filter by `YEAR`, Spark reads **only the matching partition directories** - confirmable in the physical plan's `PartitionFilters`.

In [26]:
pruned = flights_parq.filter('YEAR = 2022')
print('=== Partition pruning plan ===')
pruned.explain(True)

t0 = time.perf_counter(); n = pruned.count(); pruned_t = time.perf_counter() - t0
t0 = time.perf_counter(); n_all = flights_parq.count(); all_t = time.perf_counter() - t0
print(f'Pruned (YEAR=2022): {pruned_t:.3f}s,  rows={n}')
print(f'Full scan:          {all_t:.3f}s,    rows={n_all}')
PERF.append({'query': 'Partition_pruned',   'api': 'DataFrame', 'seconds': round(pruned_t, 3), 'rows': n})
PERF.append({'query': 'Partition_fullscan', 'api': 'DataFrame', 'seconds': round(all_t, 3),    'rows': n_all})

=== Partition pruning plan ===
== Parsed Logical Plan ==
'Filter ('YEAR = 2022)
+- Relation [FL_DATE#4402,AIRLINE#4403,AIRLINE_DOT#4404,AIRLINE_CODE#4405,DOT_CODE#4406,FL_NUMBER#4407,ORIGIN#4408,ORIGIN_CITY#4409,DEST#4410,DEST_CITY#4411,CRS_DEP_TIME#4412,DEP_TIME#4413,DEP_DELAY#4414,TAXI_OUT#4415,WHEELS_OFF#4416,WHEELS_ON#4417,TAXI_IN#4418,CRS_ARR_TIME#4419,ARR_TIME#4420,ARR_DELAY#4421,CANCELLED#4422,CANCELLATION_CODE#4423,DIVERTED#4424,CRS_ELAPSED_TIME#4425,ELAPSED_TIME#4426,... 10 more fields] parquet

== Analyzed Logical Plan ==
FL_DATE: date, AIRLINE: string, AIRLINE_DOT: string, AIRLINE_CODE: string, DOT_CODE: int, FL_NUMBER: int, ORIGIN: string, ORIGIN_CITY: string, DEST: string, DEST_CITY: string, CRS_DEP_TIME: int, DEP_TIME: double, DEP_DELAY: double, TAXI_OUT: double, WHEELS_OFF: double, WHEELS_ON: double, TAXI_IN: double, CRS_ARR_TIME: int, ARR_TIME: double, ARR_DELAY: double, CANCELLED: double, CANCELLATION_CODE: string, DIVERTED: double, CRS_ELAPSED_TIME: double, ELAPSED_TI

### 4.4 Scalability - vary shuffle partitions

In [27]:
scale_results = []
for n_parts in [8, 50, 200, 400]:
    spark.conf.set('spark.sql.shuffle.partitions', n_parts)
    t0 = time.perf_counter()
    q3_df().count()
    dt = time.perf_counter() - t0
    scale_results.append((n_parts, round(dt, 3)))
    print(f'shuffle.partitions={n_parts:>4}  ->  Q3 in {dt:.3f}s')

spark.conf.set('spark.sql.shuffle.partitions', 200)  # restore

shuffle.partitions=   8  ->  Q3 in 1.748s
shuffle.partitions=  50  ->  Q3 in 1.717s
shuffle.partitions= 200  ->  Q3 in 3.401s
shuffle.partitions= 400  ->  Q3 in 2.046s


---

## 5. Performance Comparison Table

In [28]:
import pandas as pd
perf_df = pd.DataFrame(PERF)
print(perf_df.to_string(index=False))
perf_df.to_csv('performance_results.csv', index=False)

pivot = (perf_df[perf_df['api'].isin(['RDD', 'DataFrame', 'SQL'])]
            .pivot_table(index='query', columns='api', values='seconds', aggfunc='min'))
print('\n=== RDD vs DataFrame vs SQL (seconds) ===')
print(pivot.to_string())
pivot.to_csv('performance_pivot.csv')

             query       api  seconds      rows
         Q1_filter       RDD  140.531   28379.0
         Q1_filter DataFrame   10.646   28379.0
         Q1_filter       SQL   10.374   28379.0
         Q1_filter       RDD   36.832   28379.0
         Q1_filter DataFrame   10.848   28379.0
         Q1_filter       SQL   11.997   28379.0
      Q2_aggregate       RDD   65.500      18.0
      Q2_aggregate DataFrame    9.774      18.0
      Q2_aggregate       SQL    7.547      18.0
    Q3_multi_group       RDD   54.288   24911.0
    Q3_multi_group DataFrame   13.896   24939.0
    Q3_multi_group       SQL   14.507   24911.0
   Q4_top20_routes       RDD   49.753      20.0
   Q4_top20_routes DataFrame   12.398      20.0
   Q4_top20_routes       SQL   11.207      20.0
     Q5_moving_avg       RDD   54.472   28336.0
     Q5_moving_avg DataFrame   13.827   28336.0
     Q5_moving_avg       SQL   13.094   28336.0
     Q6_cumulative       RDD   49.895   28355.0
     Q6_cumulative DataFrame   12.091   

In [ ]:
spark.stop()

---

## 6. Final Insights

### Analytical findings
1. **Delay root causes differ by airline** - low-cost carriers skew toward *late-aircraft* propagation, majors skew toward *NAS* (air-traffic / airport).
2. **Winter months** (Dec-Feb) significantly raise severe-delay probability on long-haul (>500 mi) non-cancelled flights (Q1).
3. **Cumulative cancellations** (Q6) reveal step-function spikes aligning with COVID-19 (2020 Q2) and 2022 holiday weather events.
4. **Anomaly routes** (Q12) are concentrated in weather-exposed hubs (ORD, EWR, LGA).

### Spark lessons learned
| Observation | Takeaway |
|---|---|
| DataFrame/SQL often **~3-10x faster** than RDD on identical queries | Catalyst + Tungsten eliminate serialization overhead & apply column-pruning / predicate pushdown |
| Parquet Q3 beats CSV Q3 typically **5-20x** | Columnar read + predicate pushdown + smaller I/O |
| Cached `flights` cuts Q2 wall-clock ~**3-6x** on re-runs | First pass pays columnar encode cost, later passes are in-memory |
| Broadcast join (Q9) vs sort-merge join (Q10) plan swap is **driven by table size** | Stay under `spark.sql.autoBroadcastJoinThreshold` to avoid two shuffles |
| Partition pruning on Parquet (`YEAR=2022`) scans 1 directory, not 5 | Always partition by a high-cardinality-enough, filter-frequent column |
| `shuffle.partitions` too small -> stragglers, too large -> task overhead | Rule-of-thumb: target ~128 MB per partition post-shuffle |

### Why Structured APIs > RDDs
- **Catalyst Optimizer** rewrites logical plans (predicate pushdown, constant folding, column pruning) - RDDs are opaque.
- **Tungsten** runs off-heap, code-generates whole-stage JVM bytecode, and uses cache-aware binary layout. RDDs serialize Python objects per row.
- **Auto broadcast / AQE** dynamically adapts join strategy from runtime statistics - impossible with raw RDD joins.